# TS2Vec vs TSPulse: Learned Time-Series Retrieval

This notebook compares the two first-party learned-series providers through the same DuckPD query and corpus path. It measures their contracts, preparation and bounded inference behavior, exact cosine neighborhoods, repeatability, and response to a controlled input change.

It does **not** declare a model-quality winner. The checked-in market data is synthetic, TSPulse is a published univariate checkpoint, and the local TS2Vec bundle is trained on three channels. There are no financial relevance labels here.

## 1. Environment and comparison protocol

Install both optional runtimes before selecting this environment as the notebook kernel:

```bash
uv sync --frozen --group dev --extra tspulse --extra ts2vec
```

The comparison fixes what can be fixed: both providers receive 512 observations, use the same 128 candidate endpoints, the same query endpoint, batch size 16, exact cosine search, and CPU execution. Their channel count, preprocessing, output dimension, checkpoint origin, and training burden remain intrinsic differences and are reported rather than hidden.

In [ ]:
from __future__ import annotations

import subprocess
import sys
from datetime import datetime, timedelta
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pandas

import duckpd as pd

DEMO_DIR = Path("demo") if Path("demo").is_dir() else Path(".")
DATASET = DEMO_DIR / "data" / "market-data-smoke.parquet"
TS2VEC_BUNDLE = DEMO_DIR / ".tmp" / "ts2vec-market-smoke"
TICKER = "NVDA"
WINDOW = 512
INPUT_POINTS = 640
CANDIDATE_COUNT = INPUT_POINTS - WINDOW
BATCH_SIZE = 16
TOP_K = 10
CUTOFF = datetime(2020, 1, 1) + timedelta(seconds=INPUT_POINTS)

if not DATASET.is_file():
    raise FileNotFoundError(
        f"Missing {DATASET}. Run: uv run python demo/generate_market_data.py smoke"
    )

print(f"DuckPD {pd.__version__}")
print(f"Dataset: {DATASET}")
print(f"Common workload: {CANDIDATE_COUNT} endpoints, {WINDOW} observations, batch {BATCH_SIZE}")

## 2. Create or reuse the local TS2Vec artifact

TSPulse uses its immutable published search checkpoint. TS2Vec has no equivalent pretrained finance artifact, so the repository producer trains a local bundle. The producer streams a bounded number of points per ticker, uses chronological embargoed splits, fits standardization on training points only, trains the hierarchical contrastive objective, and exports averaged safetensors plus an attestation manifest.

The default run is intentionally small enough for a CPU demonstration. Its synthetic output proves the artifact and execution path, not financial usefulness. The bundle is retained under `demo/.tmp/` so later notebook runs do not retrain it.

In [ ]:
training_seconds = None
if not TS2VEC_BUNDLE.is_dir():
    started = perf_counter()
    subprocess.run(
        [sys.executable, str(DEMO_DIR / "train_ts2vec.py")],
        check=True,
    )
    training_seconds = perf_counter() - started

print(f"TS2Vec bundle: {TS2VEC_BUNDLE}")
print(
    "Training in this run: "
    + ("reused existing bundle" if training_seconds is None else f"{training_seconds:.3f} s")
)

## 3. Inspect the contracts before loading either model

Both helpers produce immutable representation specifications. TSPulse fixes one target channel and internal affine RevIN. The TS2Vec specification is derived from the verified local manifest and includes its ordered three-channel schema and training-fitted standardization recipe. Neither permits extra outer normalization or final unit normalization.

In [ ]:
tspulse_model = pd.tspulse_series_embedding_model("bar_return")
ts2vec_model = pd.ts2vec_series_embedding_model(TS2VEC_BUNDLE)

if tspulse_model.input_length != WINDOW or ts2vec_model.input_length != WINDOW:
    raise ValueError("This comparison requires 512-observation TSPulse and TS2Vec inputs")
if ts2vec_model.input_channels != ("close_return", "bar_return", "intrabar_range"):
    raise ValueError(f"Unexpected TS2Vec channels: {ts2vec_model.input_channels}")

tspulse_representation = pd.series_representation(
    window=WINDOW,
    channels=tspulse_model.input_channels,
    sampling="observations",
    data_contract="demo/ohlc-bar-return/comparison-v1",
    encoder=tspulse_model,
)
ts2vec_representation = pd.series_representation(
    window=WINDOW,
    channels=ts2vec_model.input_channels,
    sampling="observations",
    data_contract="demo/synthetic-ohlc/ts2vec-v1",
    encoder=ts2vec_model,
)

contracts = pandas.DataFrame(
    [
        {
            "provider": "TSPulse",
            "artifact origin": "published immutable search checkpoint",
            "channels": ", ".join(tspulse_model.input_channels),
            "internal preprocessing": tspulse_model.input_normalization,
            "pooling/readout": tspulse_model.pooling,
            "dimension": tspulse_model.dimension,
        },
        {
            "provider": "TS2Vec",
            "artifact origin": "locally trained averaged safetensors",
            "channels": ", ".join(ts2vec_model.input_channels),
            "internal preprocessing": ts2vec_model.input_normalization,
            "pooling/readout": ts2vec_model.pooling,
            "dimension": ts2vec_model.dimension,
        },
    ]
).set_index("provider")
contracts

## 4. Prepare both CPU providers explicitly

Preparation is measured separately from corpus execution. TSPulse verifies/downloads its pinned files through its cache. TS2Vec re-verifies the local manifest and weight bytes, reconstructs the pinned encoder, and loads only safetensors. First-run TSPulse time may include network transfer; later runs normally reuse its verified cache.

In [ ]:
for old_session in globals().get("sessions", {}).values():
    old_session.close()

sessions = {"TSPulse": pd.connect(), "TS2Vec": pd.connect()}
preparation_seconds = {}

started = perf_counter()
tspulse_prepared = sessions["TSPulse"].prepare_series_embedding_model(tspulse_model)
preparation_seconds["TSPulse"] = perf_counter() - started

started = perf_counter()
ts2vec_prepared = sessions["TS2Vec"].prepare_series_embedding_model(
    ts2vec_model,
    artifact_dir=TS2VEC_BUNDLE,
)
preparation_seconds["TS2Vec"] = perf_counter() - started

prepared = {"TSPulse": tspulse_prepared, "TS2Vec": ts2vec_prepared}
pandas.DataFrame(
    [
        {
            "provider": name,
            "prepare seconds": preparation_seconds[name],
            "execution provider": info.execution_providers[0],
            "runtime": dict(info.runtime_versions),
        }
        for name, info in prepared.items()
    ]
).set_index("provider")

## 5. Build matched DuckPD workloads

Both plans scan the first 640 NVDA bars. `close_return` is undefined on the first row, so the TSPulse plan removes that row before constructing its 512-value `bar_return` window. This makes both providers consume rows 1–512 for the first candidate and produce exactly the same 128 endpoint timestamps. Complete input windows are filtered before `embed_series()` so a learned expression is not repeated in both a null predicate and the projection. DuckPD then performs model calls in bounded batches of 16 complete Arrow rows.

In [ ]:
def common_features(session):
    prices = session.read_parquet(DATASET, order_by=["ticker", "datetime"])
    bounded = prices[(prices["ticker"] == TICKER) & (prices["datetime"] < CUTOFF)]
    return bounded.assign(
        close_return=lambda frame: frame["close"].pct_change(),
        bar_return=lambda frame: (frame["close"] - frame["open"]) / frame["open"],
        intrabar_range=lambda frame: (frame["high"] - frame["low"]) / frame["open"],
    )


def build_tspulse_workload(session):
    features = common_features(session)
    complete = features[features["close_return"].notna()]
    query_frame = complete[["datetime", "bar_return"]].head(WINDOW)
    query = {"bar_return": tuple(float(value) for value in query_frame["bar_return"])}
    windows = complete.assign(
        bar_return_window=lambda frame: frame["bar_return"].rolling(WINDOW).to_array()
    )
    complete_windows = windows[windows["bar_return_window"].notna()]
    embedded = complete_windows.embed_series(
        columns={"bar_return": "bar_return_window"},
        into="embedding",
        representation=tspulse_representation,
        batch_size=BATCH_SIZE,
    )
    return embedded, query, query_frame.iloc[-1]["datetime"]


def build_ts2vec_workload(session):
    features = common_features(session)
    complete = features[features["close_return"].notna()]
    query_frame = complete[["datetime", *ts2vec_model.input_channels]].head(WINDOW)
    query = {
        channel: tuple(float(value) for value in query_frame[channel])
        for channel in ts2vec_model.input_channels
    }
    windows = features.assign(
        close_return_window=lambda frame: frame["close_return"].rolling(WINDOW).to_array(),
        bar_return_window=lambda frame: frame["bar_return"].rolling(WINDOW).to_array(),
        intrabar_range_window=lambda frame: frame["intrabar_range"].rolling(WINDOW).to_array(),
    )
    complete_windows = windows[
        windows["close_return_window"].notna()
        & windows["bar_return_window"].notna()
        & windows["intrabar_range_window"].notna()
    ]
    embedded = complete_windows.embed_series(
        columns={
            "close_return": "close_return_window",
            "bar_return": "bar_return_window",
            "intrabar_range": "intrabar_range_window",
        },
        into="embedding",
        representation=ts2vec_representation,
        batch_size=BATCH_SIZE,
    )
    return embedded, query, query_frame.iloc[-1]["datetime"]


workloads = {
    "TSPulse": build_tspulse_workload(sessions["TSPulse"]),
    "TS2Vec": build_ts2vec_workload(sessions["TS2Vec"]),
}
assert workloads["TSPulse"][2] == workloads["TS2Vec"][2]
print(f"Shared query endpoint: {workloads['TSPulse'][2]}")

## 6. Profile bounded corpus embedding

`profile()` executes each lazy corpus plan once and reports DuckPD execution plus provider metrics. `series_inference_seconds` is measured around provider calls; `execution_seconds` also includes scan, feature expressions, window construction, Arrow conversion, scatter-back, and result handling. Treat these as workstation observations, not stable product benchmarks.

In [ ]:
profiles = {}
performance_rows = []
for name, (frame, _, _) in workloads.items():
    profile = frame[["datetime", "embedding"]].profile()
    metrics = profile.embedding_metrics
    if metrics is None:
        raise RuntimeError(f"Missing embedding metrics for {name}")
    profiles[name] = profile
    performance_rows.append(
        {
            "provider": name,
            "prepare seconds": preparation_seconds[name],
            "profile execution seconds": profile.execution_seconds,
            "provider inference seconds": metrics["series_inference_seconds"],
            "provider calls": metrics["series_corpus_provider_calls"],
            "provider rows": metrics["series_corpus_rows"],
            "max batch rows": metrics["series_max_provider_batch_rows"],
            "Arrow bytes": metrics["series_arrow_bytes"],
            "vector bytes/row": prepared[name].dimension * 4,
        }
    )

performance = pandas.DataFrame(performance_rows).set_index("provider")
assert (performance["provider rows"] == CANDIDATE_COUNT).all()
assert (performance["max batch rows"] <= BATCH_SIZE).all()
performance

## 7. Compare exact cosine neighborhoods

Distances are meaningful only within their own representation space, so the notebook never compares a TSPulse distance numerically with a TS2Vec distance. It compares ranked endpoint identities instead. The query-by-example must rank itself first for both providers. Overlap among the remaining endpoints measures neighborhood agreement, not correctness.

In [ ]:
retrieval_results = {}
retrieval_seconds = {}
for name, (frame, query, query_endpoint) in workloads.items():
    representation = tspulse_representation if name == "TSPulse" else ts2vec_representation
    search = frame.vector.search_series(
        query,
        column="embedding",
        representation=representation,
        metric="cosine",
        k=TOP_K,
        tie_breaker="datetime",
    )[["datetime", "close", "_distance"]]
    started = perf_counter()
    result = search.collect()
    retrieval_seconds[name] = perf_counter() - started
    if result.iloc[0]["datetime"] != query_endpoint:
        raise AssertionError(f"{name} did not retrieve its query endpoint first")
    retrieval_results[name] = result

ranked_neighbors = pandas.DataFrame(
    {
        "rank": np.arange(1, TOP_K + 1),
        "TSPulse endpoint": retrieval_results["TSPulse"]["datetime"].to_numpy(),
        "TSPulse cosine distance": retrieval_results["TSPulse"]["_distance"].to_numpy(),
        "TS2Vec endpoint": retrieval_results["TS2Vec"]["datetime"].to_numpy(),
        "TS2Vec cosine distance": retrieval_results["TS2Vec"]["_distance"].to_numpy(),
    }
)

tspulse_neighbors = set(ranked_neighbors["TSPulse endpoint"].iloc[1:])
ts2vec_neighbors = set(ranked_neighbors["TS2Vec endpoint"].iloc[1:])
neighbor_jaccard = len(tspulse_neighbors & ts2vec_neighbors) / len(
    tspulse_neighbors | ts2vec_neighbors
)
print(f"Warm exact-search wall time: {retrieval_seconds}")
print(f"Top-{TOP_K - 1} non-self endpoint Jaccard: {neighbor_jaccard:.3f}")
ranked_neighbors

## 8. Compare repeatability and one controlled sensitivity

Each raw query is encoded twice to check exact repeatability on this CPU runtime. The sensitivity probe then doubles only `bar_return`. TSPulse applies per-window affine RevIN, while TS2Vec applies fixed training-set standardization and also sees two unchanged channels. A difference here explains model invariance; it is not a quality score.

In [ ]:
def cosine_distance(left, right):
    left_values = np.asarray(left, dtype=np.float32)
    right_values = np.asarray(right, dtype=np.float32)
    return float(
        1.0
        - np.dot(left_values, right_values)
        / (np.linalg.norm(left_values) * np.linalg.norm(right_values))
    )


sensitivity_rows = []
for name, (_, query, _) in workloads.items():
    session = sessions[name]
    representation = tspulse_representation if name == "TSPulse" else ts2vec_representation
    first = session.embed_series_query(query, representation=representation)
    second = session.embed_series_query(query, representation=representation)
    np.testing.assert_array_equal(first.values, second.values)

    changed_query = dict(query)
    changed_query["bar_return"] = tuple(2.0 * value for value in query["bar_return"])
    changed = session.embed_series_query(changed_query, representation=representation)
    sensitivity_rows.append(
        {
            "provider": name,
            "repeatability": "bit-identical",
            "cosine distance after 2x bar_return": cosine_distance(first.values, changed.values),
        }
    )

pandas.DataFrame(sensitivity_rows).set_index("provider")

## 9. Interpretation and next qualification step

**How they work**

- **TSPulse** is operationally simple for this contract: a fixed published 512-point univariate search checkpoint, internal affine RevIN, a fixed decoder/register readout, and 240 outputs. It requires no local training but cannot represent joint channel interaction.
- **TS2Vec** is application-owned: a local contrastive-training producer chooses channels and output width, exports averaged weights, and the provider applies training-fitted standardization followed by temporal convolutions and full-series max pooling. It supports joint multivariate input but adds training, provenance, and artifact-management responsibility.

**What the tables can establish now**

- Preparation and inference cost on this machine.
- Bounded provider calls and fixed output/storage cost.
- Deterministic query encoding on this CPU runtime.
- Whether the two spaces return similar endpoint neighborhoods.
- A concrete normalization/invariance difference.

**What they cannot establish**

- Financial relevance, predictive utility, robustness across regimes, or superiority over native/PCA/statistical/DTW controls.
- A fair learned-model winner while inputs and dimensions differ.

A qualification notebook should next use real point-in-time data, predeclared relevance labels and leakage-safe splits, matched channel and output budgets where possible, multiple TS2Vec seeds, strong conventional controls, and separate retrieval and downstream predictive metrics.

In [ ]:
for session in sessions.values():
    session.close()
print("Sessions closed. The verified TS2Vec bundle remains available for reuse.")